In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    make_scorer,
    precision_recall_fscore_support
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from datetime import timedelta
from itertools import combinations

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.feature_selection import SequentialFeatureSelector


In [15]:
def build_model(model_type: str, use_early_stopping: bool = True):
    """
    Factory function to build a model by type.
    """
    model_type = model_type.lower()

    if model_type == "xgb":
        return xgb.XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=10,
            eval_metric="aucpr",  # PR-AUC for imbalanced data
            n_jobs=-1,
            random_state=42,
            early_stopping_rounds=50 if use_early_stopping else None,
        )

    elif model_type == "logreg":
        # Pipeline: scale features, then logistic regression
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="l2",
                    C=1.0,
                    class_weight="balanced",
                    max_iter=5000,
                    solver="lbfgs",
                    n_jobs=-1,
                ))
            ]
        )

    elif model_type == "et":   # Extra Trees
        return ExtraTreesClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=5,
            n_jobs=-1,
            class_weight="balanced",
            random_state=42,
        )

    elif model_type == "rf":
        return RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=10,
            n_jobs=-1,
            class_weight="balanced",
            random_state=42,
        )

    elif model_type == "hgb":
        return HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=None,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.0,
            max_iter=300,
            class_weight="balanced",
            random_state=42,
        )

    elif model_type == "svm":
        # Scale -> LinearSVC -> Calibrated for predict_proba
        base_svm = LinearSVC(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
        )
        calibrated_svm = CalibratedClassifierCV(
            base_svm,
            method="sigmoid",  # Platt scaling
            cv=3
        )

        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("svm", calibrated_svm),
            ]
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")


In [16]:
def build_feature_sets(df_train):
    """
    Build feature set definitions using 4 base groups:
    - static
    - recency
    - history
    - trend

    Then generate ALL non-empty combinations of these groups:
    (4 choose 1) + (4 choose 2) + (4 choose 3) + (4 choose 4) = 15.

    Returns
    -------
    feature_sets : dict
        {feature_set_name: [list_of_columns], ...}
    """
    ignore_cols = ['userId', 'target_churn', 'anchor_date', 'ts_date']
    all_features = [c for c in df_train.columns if c not in ignore_cols]

    # ---- 1. Build the 4 base groups ----

    # By prefix
    recency_prefix = "recent_"
    history_prefix = "history_"
    trend_prefix = "trend_"

    recency_features = [c for c in all_features if c.startswith(recency_prefix)]
    history_features = [c for c in all_features if c.startswith(history_prefix)]
    trend_features = [c for c in all_features if c.startswith(trend_prefix)]

    # Recency also includes this explicitly
    if "days_since_last_action" in all_features:
        recency_features.append("days_since_last_action")

    # Static = everything that is NOT recency/history/trend
    static_features = [
        c for c in all_features
        if c not in recency_features
        and c not in history_features
        and c not in trend_features
    ]

    base_groups = {
        "static": sorted(static_features),
        "recency": sorted(recency_features),
        "history": sorted(history_features),
        "trend": sorted(trend_features),
    }

    # ---- 2. Generate ALL non-empty combinations of these 4 groups ----

    feature_sets = {}

    group_names = list(base_groups.keys())

    for r in range(1, len(group_names) + 1):  # r = 1,2,3,4
        for combo in combinations(group_names, r):
            # combo is a tuple like ('static',) or ('static','recency') etc.
            name = "__".join(combo)  # e.g. "static", "static__recency", ...

            # union of all columns in the selected groups
            cols = set()
            for g in combo:
                cols.update(base_groups[g])

            cols = sorted(cols)

            # skip empty sets (paranoid check)
            if len(cols) == 0:
                continue

            feature_sets[name] = cols

    # Optional: add a friendly alias for everything
    feature_sets["all_features"] = sorted(all_features)

    return feature_sets


In [17]:
def evaluate_model_cv(X, y, groups, model_type: str, n_splits=5, threshold=0.5, plot_confusion=False):
    """
    Run GroupKFold CV for a given model + features, and return aggregated metrics.
    """
    gkf = GroupKFold(n_splits=n_splits)

    oof_pred_proba = np.zeros(len(y))
    oof_true = np.array(y)

    fold_roc = []
    fold_ap = []

    for i, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"  Fold {i + 1}/{n_splits} [{model_type}]")

        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]

        clf = build_model(model_type, use_early_stopping=True)

        if model_type == "xgb":
            clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        else:
            clf.fit(X_tr, y_tr)

        preds_proba = clf.predict_proba(X_val)[:, 1]
        oof_pred_proba[val_idx] = preds_proba

        fold_roc.append(roc_auc_score(y_val, preds_proba))
        fold_ap.append(average_precision_score(y_val, preds_proba))

    # Global metrics from OOF predictions
    roc = roc_auc_score(oof_true, oof_pred_proba)
    ap = average_precision_score(oof_true, oof_pred_proba)

    pred_labels = (oof_pred_proba >= threshold).astype(int)

    prec = precision_score(oof_true, pred_labels)
    rec = recall_score(oof_true, pred_labels)
    f1 = f1_score(oof_true, pred_labels)
    cm = confusion_matrix(oof_true, pred_labels)
    tn, fp, fn, tp = cm.ravel()

    print(f"  Mean ROC-AUC: {np.mean(fold_roc):.4f} | Global ROC-AUC: {roc:.4f}")
    print(f"  Mean AP (PR-AUC): {np.mean(fold_ap):.4f} | Global AP: {ap:.4f}")
    print(f"  Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print(f"  Confusion Matrix (OOF, thr={threshold}): TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    if plot_confusion:
        plt.figure(figsize=(5, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Pred: No Churn', 'Pred: Churn'],
            yticklabels=['True: No Churn', 'True: Churn']
        )
        plt.title(f'Confusion Matrix (OOF) - {model_type}')
        plt.ylabel('True label')
        plt.xlabel('Predicted label')
        plt.tight_layout()
        plt.show()

    metrics = {
        "roc_auc": roc,
        "pr_auc": ap,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

    return metrics


In [18]:
# --- CONFIGURATION ---
# This is where you adjust the model's sensitivity
CONFIG = {
    'WINDOW_RECENT_DAYS': 7,    # Short-term period (recent trend)
    'WINDOW_HISTORY_DAYS': 14,  # Medium-term period (for comparison)
    'TARGET_WINDOW': 10,        # Prediction window (churn in the next X days)
    'STEP_SIZE': 7,             # We advance by 7 days with each iteration (Data Augmentation)
    'CHURN_EVENT': 'Cancellation Confirmation'
}

In [19]:
def load_and_clean_data(csv_path):
    """
    Loads the data and converts the temporal types.
    """
    print("Loading and cleaning the data...")
    df = pd.read_parquet(csv_path)

    df.drop("ts", axis=1)

    # On supprime les lignes sans userId valide (ex: utilisateurs non loggués)
    df = df[df['userId'].notna()]

    # Converting categorical columns to optimize memory
    categorical_cols = ['gender', 'level', 'page', 'method']
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')

    print(f"Data ready: {df.shape[0]} lines, from {df['time'].min()} to {df['time'].max()}")
    return df

In [20]:
df=load_and_clean_data("../../../data/churn-prediction-25-26/train.parquet")

Loading and cleaning the data...
Data ready: 17499636 lines, from 2018-10-01 00:00:01 to 2018-11-20 00:00:00


In [21]:
def device_type(s: str):
    if "Windows" in s:
        return "Windows"
    elif "Macintosh" in s:
        return "Macintosh"
    elif "Linux" in s:
        return "Linux"
    elif "iPad" in s:
        return "iPad"
    elif "iPhone" in s:
        return "iPhone"
    else:
        return "Different Device"


def exact_device_type(df: pd.DataFrame):
    """
    From a dataframe with at least ['userId', 'userAgent'],
    extract the device type and return a wide one-hot table by user.
    """
    df = df.copy()

    #Extract the part inside parentheses from the userAgent string
    df["device_used"] = df["userAgent"].apply(
        lambda x: x[x.index("(") : x.index(")") + 1]
        if isinstance(x, str) and "(" in x and ")" in x
        else None
    )
    # Map to simplified device categories
    df["exact_device"] = df["device_used"].apply(
        lambda x: device_type(x)
        if isinstance(x, str) else None)

    df_a = df[["userId", "exact_device"]]
    df_b = df_a.drop_duplicates().copy()

    df_b.loc[:, "val"] = 1

    # Pivot to get one column per device type
    df_c = df_b.pivot(index="userId", columns="exact_device", values="val")

    df_c = df_c.fillna(0.0)

    return df_c


def compute_features_at_anchor(df_window, anchor_date):
    """
    Compute features for all users who are active at a specific date (anchor_date).
    """
    # 1. Time slicing (Past only!)
    # "Recent" period: [Anchor - WINDOW_RECENT_DAYS, Anchor]
    start_recent = anchor_date - timedelta(days=CONFIG['WINDOW_RECENT_DAYS'])

    # "Historical" period: [Anchor - (WINDOW_HISTORY_DAYS + WINDOW_RECENT_DAYS), Anchor - WINDOW_RECENT_DAYS]
    # (we avoid overlap)
    start_history = start_recent - timedelta(days=CONFIG['WINDOW_HISTORY_DAYS'])

    # Global filtering (keep only rows that are needed for feature computation)
    df_past = df_window[
        (df_window['time'] >= start_history) & (df_window['time'] <= anchor_date)
    ].copy()

    if df_past.empty:
        return None

    # Split into two subsets to compare trends
    mask_recent = df_past['time'] >= start_recent
    df_recent = df_past[mask_recent]
    df_history = df_past[~mask_recent]

    # --- A. STATIC FEATURES & STATE (Last known value) ---
    # Take the last row of each user to get their current state (Paid/Free, etc.)
    last_status = df_past.sort_values('time').groupby('userId').last()

    features = pd.DataFrame(index=df_past['userId'].unique())

    # Simple encoding: Level (1 if paid, 0 if free) - Only if the column exists
    if 'level' in last_status.columns:
        features['is_paid'] = (last_status['level'] == 'paid').astype(int)

    # Gender (1 if F, 0 if M for example)
    if 'gender' in last_status.columns:
        features['gender_F'] = (last_status['gender'] == 'F').astype(int)

    # Device type (one-hot by user) from userAgent if available
    if 'userAgent' in df_past.columns:
        # Use all past logs so we capture any device they used up to the anchor date
        device_features = exact_device_type(df_past[['userId', 'userAgent']])
        # Join on userId index
        features = features.join(device_features, how='left')

    # Recency: Days since the very last action
    last_action_date = df_past.groupby('userId')['time'].max()
    features['days_since_last_action'] = (
        (anchor_date - last_action_date).dt.total_seconds() / (3600 * 24)
    )

    # --- B. ACTIVITY FEATURES (Counts) ---
    # Helper function to avoid repetition
    def get_counts(df_sub, prefix):
        counts = df_sub.groupby('userId').agg({
            'page': 'count',      # Total actions
            'song': 'count',      # Total songs
            'sessionId': 'nunique',  # Total sessions
            'status': 'nunique'   # Number of distinct status codes (errors etc.)
        }).rename(columns={
            'page': f'{prefix}_actions',
            'song': f'{prefix}_songs',
            'sessionId': f'{prefix}_sessions',
            'status': f'{prefix}_status'
        })

        # Specific events (Thumbs Up, Thumbs Down, Errors, Add to Playlist, etc.)
        events = df_sub[df_sub['page'].isin([
            'Thumbs Up',
            'Thumbs Down',
            'Error',
            'Add to Playlist',
            'Add Friend',
            'Submit Upgrade',
            'Upgrade',
            'Downgrade',
            'Submit Downgrade',
            'Roll Advert'
        ])]

        if not events.empty:
            event_counts = pd.crosstab(events['userId'], events['page']).add_prefix(f'{prefix}_')
            counts = counts.merge(event_counts, on='userId', how='left')

        return counts

    feats_recent = get_counts(df_recent, 'recent')
    feats_history = get_counts(df_history, 'history')

    # Merge (including device features that were already joined above)
    features = features.join(feats_recent, how='left').join(feats_history, how='left').fillna(0)

    # --- C. TREND FEATURES (EVOLUTION) ---
    # This is key for churn detection: is activity decreasing?
    # We compute daily averages to compare periods of different lengths

    avg_songs_recent = features['recent_songs'] / CONFIG['WINDOW_RECENT_DAYS']
    avg_songs_history = features['history_songs'] / CONFIG['WINDOW_HISTORY_DAYS']

    # Trend ratio: (Recent + epsilon) / (History + epsilon)
    # If < 1: activity is going down
    epsilon = 0.1  # To avoid division by zero
    features['trend_song_consumption'] = (avg_songs_recent + epsilon) / (avg_songs_history + epsilon)

    # Error ratio (is the user facing more bugs recently?)
    if 'recent_Error' in features.columns and 'history_Error' in features.columns:
        features['trend_error_rate'] = (features['recent_Error'] + epsilon) / (features['history_Error'] + epsilon)

    # Status ratio (is the user facing more bugs recently?)
    if 'recent_status' in features.columns and 'history_status' in features.columns:
        features['trend_status_rate'] = (features['recent_status'] + epsilon) / (features['history_status'] + epsilon)

    # Thumbs Up ratio (is the user liking more songs?)
    if 'recent_Thumbs Up' in features.columns and 'history_Thumbs Up' in features.columns:
        features['trend_thumbs_up'] = (features['recent_Thumbs Up'] + epsilon) / (features['history_Thumbs Up'] + epsilon)

    # Thumbs Down ratio (is the user liking more songs?)
    if 'recent_Thumbs Down' in features.columns and 'history_Thumbs Down' in features.columns:
        features['trend_thumbs_down'] = (features['recent_Thumbs Down'] + epsilon) / (features['history_Thumbs Down'] + epsilon)

    # Add to Playlist ratio (is the user liking more songs?)
    if 'recent_Add to Playlist' in features.columns and 'history_Add to Playlist' in features.columns:
        features['trend_add_to_playlist'] = (features['recent_Add to Playlist'] + epsilon) / (features['history_Add to Playlist'] + epsilon)

    # Add Friend ratio (is the user liking more songs?)
    if 'recent_Add Friend' in features.columns and 'history_Add Friend' in features.columns:
        features['trend_add_friend'] = (features['recent_Add Friend'] + epsilon) / (features['history_Add Friend'] + epsilon)

    # Submit Upgrade ratio (is the user liking more songs?)
    if 'recent_Submit Upgrade' in features.columns and 'history_Submit Upgrade' in features.columns:
        features['trend_submit_upgrade'] = (features['recent_Submit Upgrade'] + epsilon) / (features['history_Submit Upgrade'] + epsilon)

    # Upgrade ratio (is the user liking more songs?)
    if 'recent_Upgrade' in features.columns and 'history_Upgrade' in features.columns:
        features['trend_upgrade'] = (features['recent_Upgrade'] + epsilon) / (features['history_Upgrade'] + epsilon)

    # Downgrade ratio (is the user liking more songs?)
    if 'recent_Downgrade' in features.columns and 'history_Downgrade' in features.columns:
        features['trend_downgrade'] = (features['recent_Downgrade'] + epsilon) / (features['history_Downgrade'] + epsilon)

    # Submit Downgrade ratio (is the user liking more songs?)
    if 'recent_Submit Downgrade' in features.columns and 'history_Submit Downgrade' in features.columns:
        features['trend_Submit downgrade'] = (features['recent_Submit Downgrade'] + epsilon) / (features['history_Submit Downgrade'] + epsilon)

    # Roll Advert ratio (is the user liking more songs?)
    if 'recent_Roll Advert' in features.columns and 'history_Roll Advert' in features.columns:
        features['trend_Roll Advert'] = (features['recent_Roll Advert'] + epsilon) / (features['history_Roll Advert'] + epsilon)

    # Add the anchor date for tracking
    features['anchor_date'] = anchor_date

    return features


In [22]:
def run_feature_engineering_pipeline(df_logs):
    """
    Runs the sliding window loop and generates the full Train/Test dataset.
    """
    print("Starting Sliding Window pipeline...")

    min_date = df_logs['time'].min()
    max_date = df_logs['time'].max()

    # We start only when we have enough history
    start_anchor = min_date + timedelta(days=CONFIG['WINDOW_HISTORY_DAYS'] + CONFIG['WINDOW_RECENT_DAYS'])
    # We stop before the very end so we still have future data to build the target
    end_anchor = max_date - timedelta(days=CONFIG['TARGET_WINDOW'])

    current_date = start_anchor
    final_datasets = []

    while current_date <= end_anchor:
        print(f"Processing window for anchor date: {current_date.date()}")

        # 1. Generate features (X) for this anchor date
        # Use a wide view on the data to avoid unnecessary copies
        window_start = current_date - timedelta(days=CONFIG['WINDOW_HISTORY_DAYS'] +
                                                     CONFIG['WINDOW_RECENT_DAYS'] + 1)
        df_view = df_logs[(df_logs['time'] >= window_start) & (df_logs['time'] <= max_date)]

        features = compute_features_at_anchor(df_view, current_date)

        if features is None or features.empty:
            current_date += timedelta(days=CONFIG['STEP_SIZE'])
            continue

        # 2. Generate the target (Y) - Future
        # Look into the future [Anchor, Anchor + TARGET_WINDOW days]
        future_mask = (df_view['time'] > current_date) & \
                      (df_view['time'] <= current_date + timedelta(days=CONFIG['TARGET_WINDOW']))

        future_data = df_view[future_mask]

        # Identify churners
        churn_users = future_data[future_data['page'] == CONFIG['CHURN_EVENT']]['userId'].unique()

        features['target_churn'] = 0
        features.loc[features.index.isin(churn_users), 'target_churn'] = 1

        final_datasets.append(features)

        # Move the sliding window forward
        current_date += timedelta(days=CONFIG['STEP_SIZE'])

    # Final concatenation
    if not final_datasets:
        print("Warning: No valid window generated (dataset too short?)")
        return pd.DataFrame()

    full_dataset = pd.concat(final_datasets).reset_index().rename(columns={'index': 'userId'})

    print(f"Pipeline finished. Dataset generated with shape: {full_dataset.shape}")
    return full_dataset


In [23]:
df_train_ready = run_feature_engineering_pipeline(df)

Starting Sliding Window pipeline...
Processing window for anchor date: 2018-10-22
Processing window for anchor date: 2018-10-29
Processing window for anchor date: 2018-11-05
Pipeline finished. Dataset generated with shape: (50548, 51)


In [30]:

def evaluate_subset_ap(X, y, feature_subset, model_type, cv_splits):
    """
    Evaluate a given feature subset using GroupKFold and average_precision (AUC-PR).

    Parameters
    ----------
    X : pd.DataFrame
        Full feature matrix (all features).
    y : np.ndarray
        Target array.
    groups : array-like
        Group labels (userId).
    feature_subset : list of str
        Names of columns to use.
    model_type : str
        Passed to build_model(model_type, ...).
    cv_splits : list of (train_idx, val_idx)
        Precomputed GroupKFold splits.

    Returns
    -------
    ap : float
        Average precision (AUC-PR) across all folds (OOF).
    """
    X_sub = X[feature_subset]

    oof_proba = np.zeros(len(y))
    oof_true = y.copy()

    for train_idx, val_idx in cv_splits:
        X_tr, y_tr = X_sub.iloc[train_idx], y[train_idx]
        X_val, y_val = X_sub.iloc[val_idx], y[val_idx]

        clf = build_model(model_type, use_early_stopping=True)

        if model_type == "xgb":
            clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        else:
            clf.fit(X_tr, y_tr)

        proba = clf.predict_proba(X_val)[:, 1]
        oof_proba[val_idx] = proba

    ap = average_precision_score(oof_true, oof_proba)
    return ap


In [36]:

def evaluate_subset_metrics(X, y, feature_subset, model_type, cv_splits, threshold=0.5):
    """
    Evaluate a given feature subset with precomputed GroupKFold splits.

    Returns:
      - AP (PR-AUC)
      - ROC-AUC
      - precision, recall, F1 at a given threshold
      - confusion matrix entries (tn, fp, fn, tp)
    """
    X_sub = X[list(feature_subset)]
    y = np.asarray(y)

    oof_pred_proba = np.zeros(len(y))

    for train_idx, val_idx in cv_splits:
        clf = build_model(model_type, use_early_stopping=False)
        clf.fit(X_sub.iloc[train_idx], y[train_idx])
        proba = clf.predict_proba(X_sub.iloc[val_idx])[:, 1]
        oof_pred_proba[val_idx] = proba

    # Threshold-free metrics
    ap = average_precision_score(y, oof_pred_proba)
    roc = roc_auc_score(y, oof_pred_proba)

    # Threshold-based metrics
    y_pred = (oof_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, y_pred, average="binary", zero_division=0
    )

    return {
        "ap": ap,
        "roc_auc": roc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


In [37]:
def sequential_fs_path(
    df_train,
    model_type="logreg",
    direction="forward",
    max_features=None,
    n_splits=5,
    min_improvement=0.0,          # used for FORWARD
    threshold=0.5,
    backward_min_features=10,     # stop when <= this many features
    backward_max_drop=0.005,      # max allowed AP drop per removal
):
    """
    Run sequential feature selection (forward or backward) for a given model,
    logging metrics at each step.

    Parameters
    ----------
    df_train : pd.DataFrame
        Must contain features + 'target_churn' + 'userId'.
    model_type : str
        e.g. 'logreg', 'xgb', 'rf', ...
    direction : {'forward', 'backward'}
    max_features : int or None
        FORWARD: stop when this many features are selected (cap).
        BACKWARD: if not None, also acts as a minimum #features (in addition
                  to backward_min_features). Usually you can leave it None
                  and just use backward_min_features.
    n_splits : int
        Number of GroupKFold splits.
    min_improvement : float
        FORWARD only: stop if ΔAP < min_improvement.
    threshold : float
        Threshold for confusion-matrix metrics.
    backward_min_features : int
        BACKWARD: minimum number of features to retain.
    backward_max_drop : float
        BACKWARD: maximum allowed AP drop per removal step.
        If AP_new < AP_prev - backward_max_drop, we stop.

    Returns
    -------
    path_df : pd.DataFrame
        Columns: ['model_type', 'direction', 'step', 'n_features',
                  'ap', 'roc_auc', 'precision', 'recall', 'f1', 'features']
    """

    ignore_cols = ['userId', 'target_churn', 'anchor_date', 'ts_date']
    all_features = [c for c in df_train.columns if c not in ignore_cols]

    X = df_train[all_features]
    y = df_train["target_churn"].values
    groups = df_train["userId"].values

    gkf = GroupKFold(n_splits=n_splits)
    cv_splits = list(gkf.split(X, y, groups))

    # Default max_features for FORWARD
    if max_features is None and direction == "forward":
        max_features = len(all_features)

    # Initial selected / remaining
    selected = []
    remaining = all_features.copy()

    if direction == "backward":
        selected = all_features.copy()
        remaining = []

    path = []
    current_ap = None
    step = 0

    while True:
        step += 1

        if direction == "forward":
            # --- FORWARD: feature-count stopping ---
            if max_features is not None and len(selected) >= max_features:
                break

            best_metrics = None
            best_feature = None

            for f in remaining:
                candidate = selected + [f]
                metrics = evaluate_subset_metrics(
                    X, y, candidate, model_type, cv_splits, threshold=threshold
                )
                if (best_metrics is None) or (metrics["ap"] > best_metrics["ap"]):
                    best_metrics = metrics
                    best_feature = f

            if current_ap is not None:
                delta = best_metrics["ap"] - current_ap
                if delta < min_improvement:
                    print(f"[{model_type}][forward] Early stop at step {step}: "
                          f"ΔAP={delta:.5f} < {min_improvement}")
                    break
            else:
                delta = None

            selected.append(best_feature)
            remaining.remove(best_feature)
            current_ap = best_metrics["ap"]

        else:
            # --- BACKWARD: feature-count stopping (min features) ---
            # Combine backward_min_features and optional max_features into a single min limit
            min_features = backward_min_features
            if max_features is not None:
                min_features = max(min_features, max_features)

            if len(selected) <= min_features:
                # already at or below min_features: stop
                break

            best_metrics = None
            best_removed = None

            for f in selected:
                candidate = [ff for ff in selected if ff != f]
                metrics = evaluate_subset_metrics(
                    X, y, candidate, model_type, cv_splits, threshold=threshold
                )
                if (best_metrics is None) or (metrics["ap"] > best_metrics["ap"]):
                    best_metrics = metrics
                    best_removed = f

            if current_ap is not None:
                delta = best_metrics["ap"] - current_ap  # AP_new - AP_prev
                # Stop if AP drops by more than backward_max_drop
                if delta < -backward_max_drop:
                    print(f"[{model_type}][backward] Early stop at step {step}: "
                          f"ΔAP={delta:.5f} < -{backward_max_drop}")
                    break
            else:
                delta = None

            selected.remove(best_removed)
            remaining.append(best_removed)
            current_ap = best_metrics["ap"]

        # log this step
        path.append({
            "model_type": model_type,
            "direction": direction,
            "step": step,
            "n_features": len(selected),
            "ap": best_metrics["ap"],
            "roc_auc": best_metrics["roc_auc"],
            "precision": best_metrics["precision"],
            "recall": best_metrics["recall"],
            "f1": best_metrics["f1"],
            "features": selected.copy(),
        })

        print(f"[{model_type}][{direction}] step {step} "
              f"| n_features={len(selected)} | AP={best_metrics['ap']:.4f}")

    path_df = pd.DataFrame(path)
    return path_df


In [ ]:
"""
model_types = ["logreg", "xgb", "rf", "et", "hgb", "svm"]  # or whatever you’re using

all_paths = []

for m in model_types:
    print(f"\n=== {m} | forward ===")
    path_fwd = sequential_fs_path(
        df_train=df_train_ready,
        model_type=m,
        direction="forward",
        max_features=25,        # e.g., stop at 25 features
        n_splits=5,
        min_improvement=0.001,  # require at least +0.001 AP improvement
    )
    all_paths.append(path_fwd)

    print(f"\n=== {m} | backward ===")
    path_bwd = sequential_fs_path(
        df_train=df_train_ready,
        model_type=m,
        direction="backward",
        max_features=10,        # e.g., stop when 10 features remain
        n_splits=5,
        min_improvement=0.0,    # you can decide a rule here
    )
    all_paths.append(path_bwd)

fs_results_df = pd.concat(all_paths, ignore_index=True)
"""

all_paths = []

model_types = ["logreg", "xgb", "rf", "et", "hgb", "svm"]

for m in model_types:
    print(f"\n=== {m} | forward ===")
    path_fwd = sequential_fs_path(
        df_train=df_train_ready,
        model_type=m,
        direction="forward",
        max_features=40,       # cap at 30 features
        n_splits=5,
        min_improvement=0.00001, # require +0.001 AP per added feature
        threshold=0.5,
    )

    all_paths.append(path_fwd)

    print(f"\n=== {m} | backward ===")
    path_bwd = sequential_fs_path(
        df_train=df_train_ready,
        model_type=m,
        direction="backward",
        n_splits=5,
        backward_min_features=5,   # never go below 10 features
        backward_max_drop=0.05,    # stop if AP drops by more than 0.05
    )
    all_paths.append(path_bwd)

    fs_results_df = pd.concat(all_paths, ignore_index=True)
    fs_results_df.to_csv("forward_backward_fs.csv")

fs_results_df = pd.concat(all_paths, ignore_index=True)

"""
max_feats = 25
min_delta = 0.0   # stop when no further improvement

all_paths = []

for m in model_types:
    for direction in ["forward", "backward"]:
        print(f"\n=== {m} | {direction} ===")
        path = sequential_fs_path(
            df_train=df_train_ready,
            model_type=m,
            direction=direction,
            max_features=max_feats,
            n_splits=5,
            min_improvement=min_delta,
        )
        all_paths.append(path)

fs_results_df = pd.concat(all_paths, ignore_index=True)
"""
"""
max_feats = 25
min_delta_fwd = 0.001     # forward: must improve at least +0.001
min_delta_bwd = -0.001    # backward: can drop up to -0.001 per step

all_paths = []

for m in model_types:
    print(f"\n=== {m} | forward ===")
    path_fwd = sequential_fs_path(
        df_train=df_train_ready,
        model_type=m,
        direction="forward",
        max_features=max_feats,
        n_splits=5,
        min_improvement=min_delta_fwd,
    )
    all_paths.append(path_fwd)

    print(f"\n=== {m} | backward ===")
    path_bwd = sequential_fs_path(
        df_train=df_train_ready,
        model_type=m,
        direction="backward",
        max_features=max_feats,
        n_splits=5,
        min_improvement=min_delta_bwd,
    )
    all_paths.append(path_bwd)

fs_results_df = pd.concat(all_paths, ignore_index=True)
"""


=== logreg | forward ===
[logreg][forward] step 1 | n_features=1 | AP=0.0972
[logreg][forward] step 2 | n_features=2 | AP=0.1108
[logreg][forward] step 3 | n_features=3 | AP=0.1173
[logreg][forward] step 4 | n_features=4 | AP=0.1219
[logreg][forward] step 5 | n_features=5 | AP=0.1240
[logreg][forward] step 6 | n_features=6 | AP=0.1268
[logreg][forward] step 7 | n_features=7 | AP=0.1282
[logreg][forward] step 8 | n_features=8 | AP=0.1310
[logreg][forward] step 9 | n_features=9 | AP=0.1323
[logreg][forward] step 10 | n_features=10 | AP=0.1332
[logreg][forward] step 11 | n_features=11 | AP=0.1336
[logreg][forward] step 12 | n_features=12 | AP=0.1340
[logreg][forward] step 13 | n_features=13 | AP=0.1342
[logreg][forward] step 14 | n_features=14 | AP=0.1345
[logreg][forward] step 15 | n_features=15 | AP=0.1348
[logreg][forward] step 16 | n_features=16 | AP=0.1349
[logreg][forward] step 17 | n_features=17 | AP=0.1351
[logreg][forward] step 18 | n_features=18 | AP=0.1351
[logreg][forward] Ea

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 5))
for m in model_types:
    for d in ["forward", "backward"]:
        sub = fs_results_df[(fs_results_df["model_type"] == m) &
                            (fs_results_df["direction"] == d)]
        if sub.empty:
            continue
        label = f"{m} ({d})"
        plt.plot(sub["n_features"], sub["ap"], marker="o", label=label)

plt.xlabel("Number of Features")
plt.ylabel("AUC-PR (Average Precision)")
plt.title("Sequential Feature Selection Paths")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Example: best sparse logreg forward with <= 15 features
mask = (
    (fs_results_df["model_type"] == "logreg") &
    (fs_results_df["direction"] == "forward") &
    (fs_results_df["n_features"] <= 15)
)
best_sparse = fs_results_df[mask].sort_values("ap", ascending=False).iloc[0]
best_sparse_features = best_sparse["features"]
best_sparse_ap = best_sparse["ap"]
print(len(best_sparse_features), best_sparse_ap)

In [ ]:
def generate_kaggle_submission(
    model,
    df_logs_full,
    feature_cols,
    all_test_user_ids,
    output_file='submission.csv',
    seuil=0.5
):
    """
    Generate a BINARY Kaggle submission, with robust handling of ID types
    and users without activity in the final window.
    """
    print("\n--- Generating ROBUST submission file (Binary Mode) ---")

    # 1. Reference date (anchor)
    last_date = df_logs_full['time'].max()
    print(f"Anchor date: {last_date}")

    # 2. Compute features for users active around the anchor date
    X_test_active = compute_features_at_anchor(df_logs_full, last_date)

    if X_test_active is None:
        X_test_active = pd.DataFrame()
        print("Warning: No active users in the last window!")

    # 3. Predictions for active users
    if not X_test_active.empty:
        # Ensure all expected feature columns are present
        for col in feature_cols:
            if col not in X_test_active.columns:
                X_test_active[col] = 0

        # Keep the same column order as in training
        X_test_active = X_test_active[feature_cols]

        active_probs = model.predict_proba(X_test_active)[:, 1]

        df_preds = pd.DataFrame({
            'userId': X_test_active.index,
            'prediction': active_probs
        })
    else:
        df_preds = pd.DataFrame(columns=['userId', 'prediction'])

    # 4. Merge with the full test user list
    # Start from all test user IDs provided (e.g., from sample_submission or test.csv)
    final_submission = pd.DataFrame({'userId': all_test_user_ids})

    # --- IMPORTANT: Harmonize ID types so the merge works correctly ---
    final_submission['userId'] = final_submission['userId'].astype(str)
    df_preds['userId'] = df_preds['userId'].astype(str)

    # Remove potential duplicates to avoid exploding the number of rows
    final_submission = final_submission.drop_duplicates(subset=['userId'])
    df_preds = df_preds.drop_duplicates(subset=['userId'])

    # Left join: keep all test users, attach predictions where available
    final_submission = final_submission.merge(df_preds, on='userId', how='left')

    # 5. Handle missing predictions (users inactive in the last window)
    avg_proba = final_submission['prediction'].mean()
    if pd.isna(avg_proba):
        avg_proba = 0.5  # fallback if no predictions at all

    missing_count = final_submission['prediction'].isna().sum()
    print(
        f"Inactive users: {missing_count} "
        f"(filled with mean probability: {avg_proba:.4f})"
    )

    final_submission['prediction'] = final_submission['prediction'].fillna(avg_proba)

    # 6. Apply threshold to get binary churn label
    final_submission['is_churn'] = (final_submission['prediction'] >= seuil).astype(int)

    # 7. Save submission file
    output_df = final_submission[['userId', 'is_churn']]
    output_df.columns = ['id', 'target']

    output_df.to_csv(output_file, index=False)
    print(f"Saved: {output_file} ({len(output_df)} rows)")

    return output_df


In [ ]:
sample = pd.read_csv('../../../data/churn-prediction-25-26/example_submission.csv')
all_users_list = sample['id'].unique()
all_users_list

In [ ]:
test_set=pd.read_parquet("../../../data/churn-prediction-25-26/test.parquet")